# Bass sepanjang rekaman — eksperimen chunking

Gunakan model Base yang sudah berhasil di notebook 02. Notebook ini tidak
mengunduh ulang model atau memasang dependency lagi.

**Taruh satu rekaman gitar/piano di `data/full_song/`**, lalu Restart Kernel → Run All.
Folder dibuat oleh sel pertama, atau buat lewat file browser. Bisa WAV/MP3/M4A.
Seluruh durasi rekaman dipakai (hingga 10 menit untuk POC), termasuk lagu 3 menit.
Notebook 02 tetap untuk uji klip 10 detik.

Sumber diproses sebagai potongan 10 detik dengan overlap nominal 1 detik,
kemudian disambung memakai crossfade. Untuk lagu 180 detik ada 20 potongan;
potongan akhir menyesuaikan batas akhir rekaman. Model dimuat sekali per proses,
sehingga runtime total dapat jauh lebih lama daripada satu klip.

**Batas kualitas:** setiap chunk digenerasi mandiri. Crossfade menghaluskan
transisi audio, tetapi tidak menjamin pola bass/groove atau fase konsisten sepanjang
lagu. Dengarkan sambungan dan harmoni. Ini belum generator lagu penuh yang dilatih
khusus untuk konteks 3 menit, dan belum bukti kemampuan iPhone.

**Status kualitas:** telah dilaporkan glitch berat setelah sekitar detik 10 pada
hasil nyata pengguna. Jangan menganggap mode ini layak hanya karena tes timeline
lulus. Periksa hasil lewat notebook 04 sebelum mengulang generasi lagu penuh.


In [ ]:
from pathlib import Path
import hashlib, json, sys
from IPython.display import Audio, display
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "configs/mac-full-song.json").is_file(), "Buka dari checkout StemsGenerating-Models"
sys.path.insert(0, str(ROOT / "scripts"))
sys.path.insert(0, str(ROOT / "tools/stem_eval"))
from mac_workflow import choose_source, check_model_runtime
from setup_runtime import runtime_python
from stream_process import run_logged
from provenance import sha256
from system_tools import require_audio_tools
SONG_DATA = ROOT / "data/full_song"
SONG_DATA.mkdir(parents=True, exist_ok=True)


## 1. Pilih rekaman sendiri

`SOURCE=None` memilih satu rekaman di `data/full_song/`. Folder terpisah ini
menghindari pemilihan input.wav lama milik notebook 02. Untuk memilih path lain,
isi SOURCE dengan Path yang sesuai. DRY_RUN=True hanya menyiapkan dan merencanakan
potongan, tanpa model; default False menjalankan generasi nyata.


In [ ]:
SOURCE = None
DRY_RUN = False
require_audio_tools()
source = choose_source(SONG_DATA, SOURCE)
PROFILE_PATH = ROOT / "configs/mac-full-song.json"
PROFILE = json.loads(PROFILE_PATH.read_text())
# Same recording + profile + model manifest => same folder, allowing safe resume.
identity = sha256(source) + json.dumps(PROFILE, sort_keys=True) + sha256(ROOT / "configs/assets-base.json")
RUN = ROOT / "runs" / ("full_song_" + hashlib.sha256(identity.encode()).hexdigest()[:20])
print("Sumber:", source)
print("Profil:", json.dumps(PROFILE, indent=2))
print("Folder run/resume:", RUN)


## 2. Persiapkan seluruh lagu dan periksa timeline

Rencana mencatat durasi penuh, jumlah chunk, hash audio, dan posisi berdasarkan
frame sampel. File sumber tidak diubah. Output dan input yang berubah checksum
tidak dipakai diam-diam untuk resume. Sumber yang lebih pendek dari 10 detik
memiliki satu chunk dengan padding; final output dipotong kembali ke panjang asli.


In [ ]:
command = [str(ROOT / "scripts/run_song.py"), "--source", str(source),
           "--out", str(RUN), "--profile", str(PROFILE_PATH)]
run_logged([sys.executable, *command], check=True)
plan = json.loads((RUN / "plan.json").read_text())
print("Durasi sumber:", plan["duration"], "detik")
print("Jumlah chunk:", len(plan["jobs"]))
print("Status:", plan["status"])


## 3. Jalankan atau lanjutkan generasi

Chunk selesai disimpan dengan checksum. Jika proses terputus, jalankan notebook
lagi dengan sumber/profil sama: chunk selesai dipakai kembali, chunk gagal memakai
folder attempt baru. Jangan jalankan dua proses untuk folder run sama; lock OS
menolak generasi bersamaan. Mengubah seed/instrument membuat folder run baru.

Waktu tiap chunk ditampilkan. Model hanya dimuat bila masih ada chunk tersisa.
Jika semua chunk telah selesai, notebook langsung memverifikasi hasil dan
menyambungkannya. Tidak ada unduhan model pada langkah ini. Jika runtime/aset
belum siap, sel akan gagal dengan error; selesaikan notebook 02 terlebih dahulu.


In [ ]:
if DRY_RUN:
    print("DRY RUN: belum menjalankan model atau membuat stem pengiring")
else:
    assert runtime_python().is_file(), "Selesaikan instalasi notebook 02 terlebih dahulu"
    print(json.dumps(check_model_runtime(runtime_python()), indent=2))
    run_logged([str(runtime_python()), *command, "--execute"], check=True)


## 4. Dengarkan dan ekspor hasil lagu penuh

`bass_full.wav` adalah stem pengiring; `mix_full.wav` mencampurnya dengan sumber
bersih. Durasi output sama dengan hasil decode rekaman sumber, dengan sample rate
32 kHz stereo PCM16. Mix memakai gain 0.5 untuk sumber dan stem, tanpa normalisasi
berbeda per potongan. Periksa sambungan di sekitar detik 9, 18, 27, dan seterusnya.
Hasil tersimpan di RUN dan bisa diunduh lewat file browser JupyterLab.


In [ ]:
report_file = RUN / "experiment.json"
if report_file.exists() and json.loads(report_file.read_text()).get("status") == "generated":
    for name in (PROFILE["instrument"] + "_full.wav", "mix_full.wav"):
        path = RUN / name
        print(name, path)
        display(Audio(filename=str(path)))
else:
    print("Belum ada hasil lagu penuh yang selesai; lihat plan/status chunk di", RUN)
